# DPSR Pipeline — Doubly Permanently Shadowed Regions

### ISRO Hackathon | LOLA South Pole DEM | 15168 × 15168 @ 20 m

**Pipeline overview**

| Step | Task                                       | Output                    |
| ---- | ------------------------------------------ | ------------------------- |
| 0    | Install & imports                          | —                         |
| 1    | Load DEM                                   | elevation array           |
| 2    | Rasterize PSR shapefile                    | `PSR_mask.tif`            |
| 3    | Slope + Aspect                             | `slope.tif`, `aspect.tif` |
| 4    | Hillshade                                  | `hillshade.tif`           |
| 5    | Illumination map                           | `illumination.tif`        |
| 6    | Extract PSR coordinates                    | `psr_pixels.npy`          |
| 7    | Single ray test (debug)                    | plot                      |
| 8    | 360° horizon (single pixel)                | plot                      |
| 9    | Precompute Bresenham rays                  | ray tables                |
| 10   | DPSR extraction — CPU (Numba ×1000 faster) | `DPSR.tif`                |
| 11   | DPSR extraction — GPU (optional)           | `DPSR.tif`                |
| 12   | Final visualisation                        | summary plot              |


## Cell 0 — Install dependencies


In [ ]:
# Run once — installs missing packages into the active venv
import subprocess, sys
pkgs = ["rasterio","numpy","geopandas","matplotlib","scipy","numba"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
print("All packages ready.")


## Cell 0b — Imports & configuration


In [ ]:
import math, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from rasterio.features import rasterize
from numba import njit, prange

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ── Paths (notebook lives in output/; data is one level up) ───────────────────
BASE        = Path.cwd().parent          # ISRO_Hackathon/
DATA_DIR    = BASE / "data"
OUT_DIR     = BASE / "results"
IMG_DIR     = BASE / "images"

DEM_PATH          = DATA_DIR / "ldem_85s_20m_float.lbl"
PSR_SHP_PATH      = DATA_DIR / "LPSR_80S_20MPP_ADJ.shp"
PSR_MASK_PATH     = OUT_DIR  / "PSR_mask.tif"
SLOPE_PATH        = OUT_DIR  / "slope.tif"
ASPECT_PATH       = OUT_DIR  / "aspect.tif"
HILLSHADE_PATH    = OUT_DIR  / "hillshade.tif"
ILLUMINATION_PATH = OUT_DIR  / "illumination.tif"
DPSR_PATH         = OUT_DIR  / "DPSR.tif"

# ── Algorithm parameters ──────────────────────────────────────────────────────
CELLSIZE          = 20.0    # metres per pixel
SUN_AZIMUTH       = 0.0     # degrees 0=North 90=East; change per epoch
SUN_ELEVATION     = 1.54    # degrees — peak solar elevation at ~89.5 S latitude
N_ANGLES          = 72      # rays per pixel  (360/5 = 72)
MAX_DISTANCE      = 2500    # 2500 px x 20 m = 50 km (covers Amundsen crater)

print("Configuration ready.")
print(f"  DEM  : {DEM_PATH}")
print(f"  OUT  : {OUT_DIR}")

ANNUAL_MODE     = False  # True → sweep all azimuths (best accuracy)


---

## Step 1 — Load LOLA DEM

- Format: PDS3 (`.lbl` + `.img`)
- Resolution: 20 m/pixel
- Size: 15 168 × 15 168 pixels
- Values stored in **kilometres** → converted to **metres**


In [ ]:
print("Loading DEM …")
t0 = time.perf_counter()

with rasterio.open(DEM_PATH) as ds:
    elevation = ds.read(1, out_dtype=np.float32) * 1000.0   # km → m
    DEM_META  = ds.meta.copy()
    DEM_META.update(dtype="float32", count=1, compress="lzw")
    DEM_CRS       = ds.crs
    DEM_TRANSFORM = ds.transform

print(f"  Shape     : {elevation.shape}")
print(f"  Dtype     : {elevation.dtype}")
print(f"  RAM       : {elevation.nbytes/1e6:.0f} MB")
print(f"  Min elev  : {elevation.min():.1f} m")
print(f"  Max elev  : {elevation.max():.1f} m")
print(f"  Mean elev : {elevation.mean():.1f} m")
print(f"  CRS       : {DEM_CRS}")
print(f"  Loaded in {time.perf_counter()-t0:.2f} s")


In [ ]:
# Visualise DEM
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(elevation, cmap="terrain", interpolation="none")
plt.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.7)
ax.set_title("LOLA South Pole DEM  (15 168 × 15 168 @ 20 m)")
ax.axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "step01_dem.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 2 — Rasterize PSR Shapefile → PSR_mask.tif

Converts the LOLA PSR polygon shapefile to a binary raster aligned with the DEM.


In [ ]:
if PSR_MASK_PATH.exists():
    print(f"[SKIP] {PSR_MASK_PATH.name} already exists — loading …")
    with rasterio.open(PSR_MASK_PATH) as ds:
        psr_mask = ds.read(1)
else:
    print("Loading PSR shapefile …")
    psr = gpd.read_file(PSR_SHP_PATH)
    print(f"  Polygons : {len(psr)}")
    print(f"  PSR CRS  : {psr.crs}")

    if psr.crs is None:
        print("  No CRS found — assigning DEM CRS")
        psr = psr.set_crs(DEM_CRS)
    elif psr.crs != DEM_CRS:
        print("  Reprojecting PSR to DEM CRS …")
        psr = psr.to_crs(DEM_CRS)

    print("Rasterizing …")
    t0 = time.perf_counter()
    psr_mask = rasterize(
        [(geom, 1) for geom in psr.geometry],
        out_shape = elevation.shape,
        transform = DEM_TRANSFORM,
        fill      = 0,
        dtype     = "uint8",
    )

    out_meta = DEM_META.copy(); out_meta.update(dtype="uint8", count=1)
    with rasterio.open(PSR_MASK_PATH, "w", **out_meta) as dst:
        dst.write(psr_mask, 1)
    print(f"  Done in {time.perf_counter()-t0:.1f} s")
    print(f"  Saved  → {PSR_MASK_PATH.name}")

print(f"  PSR pixels  : {psr_mask.sum():,}")
print(f"  Coverage    : {psr_mask.sum()/psr_mask.size*100:.2f} %")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(psr_mask, cmap="gray", interpolation="none")
ax.set_title(f"PSR Mask  ({psr_mask.sum():,} pixels)")
ax.axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "step02_psr_mask.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 3 — Slope and Aspect

Computed from the DEM using NumPy gradient.

- **Slope** : steepness in degrees
- **Aspect** : compass direction of downhill slope (0–360°)


In [ ]:
if SLOPE_PATH.exists() and ASPECT_PATH.exists():
    print("[SKIP] slope.tif and aspect.tif already exist — loading …")
    with rasterio.open(SLOPE_PATH)  as ds: slope_deg = ds.read(1)
    with rasterio.open(ASPECT_PATH) as ds: aspect    = ds.read(1)
else:
    print("Computing gradient on 15k×15k DEM (may take ~30 s) …")
    t0 = time.perf_counter()
    dz_dy, dz_dx = np.gradient(elevation, CELLSIZE)

    slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))).astype(np.float32)
    aspect    = np.degrees(np.arctan2(-dz_dx, dz_dy))
    aspect    = ((aspect + 360) % 360).astype(np.float32)

    out_meta = DEM_META.copy(); out_meta.update(dtype="float32", count=1)
    with rasterio.open(SLOPE_PATH,  "w", **out_meta) as dst: dst.write(slope_deg, 1)
    with rasterio.open(ASPECT_PATH, "w", **out_meta) as dst: dst.write(aspect,    1)
    print(f"  Done in {time.perf_counter()-t0:.1f} s")

print(f"  Slope  min={slope_deg.min():.2f}°  max={slope_deg.max():.2f}°  mean={slope_deg.mean():.2f}°")
print(f"  Aspect min={aspect.min():.1f}°  max={aspect.max():.1f}°")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(slope_deg, cmap="terrain",  interpolation="none")
im1 = axes[1].imshow(aspect,    cmap="hsv",      interpolation="none")
plt.colorbar(im0, ax=axes[0], label="Slope (°)",  shrink=0.7)
plt.colorbar(im1, ax=axes[1], label="Aspect (°)", shrink=0.7)
axes[0].set_title("Slope Map");  axes[0].axis("off")
axes[1].set_title("Aspect Map"); axes[1].axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "step03_slope_aspect.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 4 — Hillshade

Simulates illumination from the Sun at a given azimuth and elevation angle.

```
Sun azimuth   = 180°  (from south)
Sun elevation = 2°    (grazing angle — typical near poles)
```

> **Note**: hillshade is used for visualisation only.
> The illumination map (Step 5) now uses solar shadow casting.


In [ ]:
# Hillshade — visualisation only; NOT used as illumination input.
if HILLSHADE_PATH.exists():
    print("[SKIP] hillshade.tif already exists — loading …")
    with rasterio.open(HILLSHADE_PATH) as ds:
        hillshade = ds.read(1)
else:
    print("Computing hillshade …")
    t0 = time.perf_counter()

    dz_dy, dz_dx = np.gradient(elevation, CELLSIZE)
    slope_hs  = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    aspect_hs = np.arctan2(-dz_dx, dz_dy)

    az_rad = math.radians(SUN_AZIMUTH)
    el_rad = math.radians(SUN_ELEVATION)

    hillshade = (
        math.sin(el_rad) * np.cos(slope_hs)
        + math.cos(el_rad) * np.sin(slope_hs) * np.cos(az_rad - aspect_hs)
    ).clip(0, 1).astype(np.float32)

    out_meta = DEM_META.copy(); out_meta.update(dtype="float32", count=1)
    with rasterio.open(HILLSHADE_PATH, "w", **out_meta) as dst:
        dst.write(hillshade, 1)
    print(f"  Done in {time.perf_counter()-t0:.1f} s  →  {HILLSHADE_PATH.name}")

print(f"  Hillshade  min={hillshade.min():.3f}  max={hillshade.max():.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(hillshade, cmap="gray", interpolation="none")
ax.set_title(f"Hillshade  (az={SUN_AZIMUTH}°  el={SUN_ELEVATION}°)")
ax.axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "step04_hillshade.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 5 — Solar Shadow Map (replaces hillshade threshold)

**Previous approach**: `illumination = (hillshade >= 0.20)`

- Hillshade is a dot product of slope normal and sun direction.
- A slope can _face_ the sun yet still be shadowed by a nearer ridge.
- Hillshade cannot detect that blocking ridge; shadow casting can.

**Current approach**: ray-cast from each pixel toward the sun.
A pixel is **in shadow** if any terrain along the ray satisfies:

```
(h_d − h_P) / dist_d  ≥  tan(SUN_ELEVATION)
```

This is the same horizon-angle derivation as the DPSR kernel.
Equivalent to GRASS `r.sunmask` and GDAL viewshed.

Set `ANNUAL_MODE = True` to sweep all azimuths (annual illumination).


In [ ]:
if not ILLUMINATION_PATH.exists():
    import math
    from numba import njit, prange
    
    # Solar shadow map — replaces hillshade > threshold
    # Pixel P is illuminated iff no terrain toward the sun exceeds sun elevation:
    #   (h_d - h_P) / dist_d  <  tan(SUN_ELEVATION)   for all d along sun ray
    # Same derivation as the DPSR LOS kernel; equivalent to GRASS r.sunmask.
    
    _sun_tan  = math.tan(math.radians(SUN_ELEVATION))
    _az_rad   = math.radians(SUN_AZIMUTH)
    _steps    = np.arange(1, MAX_DISTANCE + 1, dtype=np.float64)
    _sun_dr   = np.round(-math.cos(_az_rad) * _steps).astype(np.int32)
    _sun_dc   = np.round( math.sin(_az_rad) * _steps).astype(np.int32)
    _sun_dist = (np.sqrt(_sun_dr.astype(np.float64)**2 +
                         _sun_dc.astype(np.float64)**2) * CELLSIZE).astype(np.float32)
    _sun_dist = np.where(_sun_dist < 1.0, np.float32(1.0), _sun_dist)
    
    @njit(parallel=True, cache=True, fastmath=True)
    def _shadow_kernel(elevation, sun_dr, sun_dc, sun_dist, sun_tan):
        H, W = elevation.shape
        n    = sun_dr.shape[0]
        out  = np.ones((H, W), dtype=np.uint8)   # default: illuminated
        for row in prange(H):
            for col in range(W):
                cur_h = elevation[row, col]
                for d in range(n):
                    r = row + sun_dr[d]
                    c = col + sun_dc[d]
                    if r < 0 or r >= H or c < 0 or c >= W:
                        break
                    if (elevation[r, c] - cur_h) / sun_dist[d] >= sun_tan:
                        out[row, col] = 0
                        break
        return out

    print(f'Solar shadow  az={SUN_AZIMUTH:.0f} el={SUN_ELEVATION:.2f} deg '
          f'radius={MAX_DISTANCE*CELLSIZE/1000:.0f} km ...')
    import time as _t; _t0 = _t.time()
    illumination = _shadow_kernel(elevation, _sun_dr, _sun_dc, _sun_dist, _sun_tan)
    print(f'  done {_t.time()-_t0:.1f} s   lit={illumination.mean()*100:.1f}%')

    out_meta = DEM_META.copy()
    out_meta.update(dtype='uint8', count=1, compress='lzw')
    with rasterio.open(ILLUMINATION_PATH, 'w', **out_meta) as dst:
        dst.write(illumination, 1)
    print(f'Saved: {ILLUMINATION_PATH}')
else:
    with rasterio.open(ILLUMINATION_PATH) as ds:
        illumination = ds.read(1)
    print(f'[skip] {ILLUMINATION_PATH.name} already exists')

print(f'Illuminated: {illumination.sum():,} / {illumination.size:,}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(illumination, cmap="gray", interpolation="none")
axes[0].set_title("Illumination Map  (white = lit, black = shadow)")
axes[0].axis("off")

# Overlay on DEM
axes[1].imshow(elevation,    cmap="gray",  interpolation="none", alpha=1.0)
axes[1].imshow(illumination, cmap="Reds",  interpolation="none", alpha=0.35)
axes[1].set_title("DEM + Illumination Overlay")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(IMG_DIR / "step05_illumination.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 6 — Extract PSR Pixel Coordinates


In [ ]:
psr_rows_all, psr_cols_all = np.where(psr_mask == 1)
psr_rows_all = psr_rows_all.astype(np.int32)
psr_cols_all = psr_cols_all.astype(np.int32)

np.save(OUT_DIR / "psr_pixels.npy", np.stack([psr_rows_all, psr_cols_all], axis=1))

print(f"Total PSR pixels : {len(psr_rows_all):,}")
print(f"Saved → psr_pixels.npy")
print(f"First 5 pixels (row, col):")
for i in range(5):
    print(f"  ({psr_rows_all[i]}, {psr_cols_all[i]})")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(psr_mask, cmap="gray", interpolation="none")
ax.scatter(psr_cols_all[:5000], psr_rows_all[:5000], s=0.3, color="red", alpha=0.5)
ax.set_title(f"PSR pixels  (sample of 5,000 shown in red)")
ax.axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "step06_psr_pixels.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 7 — Single Ray Visibility Test (Debug / Visualisation)

Cast one ray from one PSR pixel to verify the algorithm visually.


In [ ]:
# Pick the first PSR pixel
test_row = int(psr_rows_all[0])
test_col = int(psr_cols_all[0])
print(f"Test pixel : row={test_row}  col={test_col}")
print(f"Elevation  : {elevation[test_row, test_col]:.1f} m")

# One ray at 45°
angle     = math.radians(45)
max_dist  = MAX_DISTANCE
cur_h     = float(elevation[test_row, test_col])

ray_rows, ray_cols = [], []
highest_tan = -1e9
visible     = False

for d in range(1, max_dist):
    r = int(test_row + d * math.sin(angle))
    c = int(test_col + d * math.cos(angle))
    if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
        break
    ray_rows.append(r); ray_cols.append(c)

    dist         = d * CELLSIZE
    terrain_tan  = (float(elevation[r, c]) - cur_h) / dist

    if illumination[r, c] == 1 and terrain_tan >= highest_tan:
        visible = True
        print(f"  Visible illuminated terrain at step d={d}  r={r}  c={c}")
        break
    if terrain_tan > highest_tan:
        highest_tan = terrain_tan

print(f"  Ray result : {'VISIBLE (not DPSR)' if visible else 'BLOCKED (DPSR candidate)'}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(elevation, cmap="gray", interpolation="none")
ax.scatter(test_col, test_row, color="red",    s=80, zorder=5, label="PSR pixel")
ax.plot(ray_cols,    ray_rows,  color="yellow", lw=1.5, label="Ray (45°)")
ax.set_xlim(test_col - 60, test_col + 60)
ax.set_ylim(test_row + 60, test_row - 60)
ax.legend(fontsize=9)
ax.set_title("Single Ray Visualisation (45°)")
plt.tight_layout()
plt.savefig(IMG_DIR / "step07_single_ray.png", dpi=120, bbox_inches="tight")
plt.show()


---

## Step 8 — 360° Horizon Analysis (Single Pixel)

Test all 72 ray directions for one PSR pixel.


In [ ]:
angles  = np.linspace(0, 2*math.pi, N_ANGLES, endpoint=False)
cur_h   = float(elevation[test_row, test_col])

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(elevation, cmap="gray", interpolation="none")
ax.scatter(test_col, test_row, color="red", s=80, zorder=5, label="PSR pixel")

pixel_visible = False

for angle in angles:
    highest_tan = -1e9
    ray_x, ray_y = [], []

    for d in range(1, MAX_DISTANCE):
        r = int(test_row + d * math.sin(angle))
        c = int(test_col + d * math.cos(angle))
        if r < 0 or r >= elevation.shape[0] or c < 0 or c >= elevation.shape[1]:
            break
        ray_y.append(r); ray_x.append(c)

        dist        = d * CELLSIZE
        terrain_tan = (float(elevation[r, c]) - cur_h) / dist

        if illumination[r, c] == 1 and terrain_tan >= highest_tan:
            pixel_visible = True
            break
        if terrain_tan > highest_tan:
            highest_tan = terrain_tan

    ax.plot(ray_x, ray_y, lw=0.4, alpha=0.6)
    if pixel_visible:
        break

result_str = "Visible illuminated terrain → NOT DPSR" if pixel_visible else "No illuminated terrain → DPSR"
ax.set_title(f"360° Horizon  |  {result_str}")
ax.set_xlim(test_col - 80, test_col + 80)
ax.set_ylim(test_row + 80, test_row - 80)
plt.tight_layout()
plt.savefig(IMG_DIR / "step08_360_horizon.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Classification : {result_str}")


---

## Step 9 — Precompute Bresenham Ray Offsets

**Why Bresenham instead of `int(d * sin(angle))`?**

- Trigonometric functions computed **once** (72 calls total), not 720 billion times
- Each pixel visited **exactly once** — no duplicates
- Pure integer arithmetic in the inner loop (→ faster)


In [ ]:
def _bresenham_ray(sin_a, cos_a, max_dist):
    end_r = int(round(max_dist * sin_a))
    end_c = int(round(max_dist * cos_a))
    abs_dr, abs_dc = abs(end_r), abs(end_c)
    sr = 1 if end_r >= 0 else -1
    sc = 1 if end_c >= 0 else -1
    dr_list, dc_list = [], []
    r = c = 0
    if abs_dc >= abs_dr:
        err = 2 * abs_dr - abs_dc
        for _ in range(min(abs_dc, max_dist)):
            c += sc
            if err >= 0: r += sr; err -= 2 * abs_dc
            err += 2 * abs_dr
            dr_list.append(r); dc_list.append(c)
    else:
        err = 2 * abs_dc - abs_dr
        for _ in range(min(abs_dr, max_dist)):
            r += sr
            if err >= 0: c += sc; err -= 2 * abs_dr
            err += 2 * abs_dc
            dr_list.append(r); dc_list.append(c)
    return dr_list, dc_list

SENTINEL = np.int32(-32767)
ray_dr   = np.full((N_ANGLES, MAX_DISTANCE), SENTINEL, dtype=np.int32)
ray_dc   = np.full((N_ANGLES, MAX_DISTANCE), SENTINEL, dtype=np.int32)
ray_dist = np.zeros((N_ANGLES, MAX_DISTANCE), dtype=np.float32)
ray_len  = np.zeros(N_ANGLES, dtype=np.int32)

for a, angle in enumerate(np.linspace(0, 2*math.pi, N_ANGLES, endpoint=False)):
    drl, dcl = _bresenham_ray(math.sin(angle), math.cos(angle), MAX_DISTANCE)
    n = min(len(drl), MAX_DISTANCE)
    ray_len[a] = n
    dr_a = np.array(drl[:n], dtype=np.int32)
    dc_a = np.array(dcl[:n], dtype=np.int32)
    ray_dr[a, :n] = dr_a
    ray_dc[a, :n] = dc_a
    d = np.sqrt(dr_a.astype(np.float64)**2 + dc_a.astype(np.float64)**2) * CELLSIZE
    ray_dist[a, :n] = np.maximum(d, 1.0).astype(np.float32)

print(f"Ray table shape   : {ray_dr.shape}")
print(f"Avg pixels/ray    : {ray_len.mean():.1f}  (DDA would be {MAX_DISTANCE})")
print(f"Iterations saved  : {(1 - ray_len.mean()/MAX_DISTANCE)*100:.1f}%")


---

## Step 10 — DPSR Extraction (CPU — Numba Parallel)

**Complexity before → after**

|                       | Before (pure Python) | After (Numba + prange) |
| --------------------- | -------------------- | ---------------------- |
| Language              | CPython interpreter  | Compiled machine code  |
| Parallelism           | 1 thread             | All CPU cores          |
| Trig in inner loop    | arctan + degrees     | **None**               |
| Redundant pixels      | ~14% (DDA)           | 0% (Bresenham)         |
| **Estimated runtime** | **40+ hours**        | **1–5 minutes**        |

**Algorithm invariant (unchanged):** visibility check runs **before** horizon update.


In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def classify_psr_pixels(elevation, illumination,
                         psr_rows, psr_cols,
                         ray_dr, ray_dc, ray_dist, ray_len):
    """
    Numba parallel DPSR kernel.
    Returns uint8 array: 1=DPSR  0=non-DPSR  for each PSR pixel.
    """
    n_psr    = psr_rows.shape[0]
    n_angles = ray_dr.shape[0]
    n_rows   = elevation.shape[0]
    n_cols   = elevation.shape[1]
    result   = np.zeros(n_psr, dtype=np.uint8)

    for i in prange(n_psr):           # parallel over all PSR pixels
        row   = psr_rows[i]
        col   = psr_cols[i]
        cur_h = elevation[row, col]
        is_dpsr = True

        for a in range(n_angles):
            highest_tan = -1.0e18
            n_steps     = ray_len[a]

            for d in range(n_steps):
                r = row + ray_dr[a, d]
                c = col + ray_dc[a, d]
                if r < 0 or r >= n_rows or c < 0 or c >= n_cols:
                    break

                terrain_tan = (elevation[r, c] - cur_h) / ray_dist[a, d]

                # Visibility check BEFORE horizon update (algorithm invariant)
                if illumination[r, c] == 1 and terrain_tan >= highest_tan:
                    is_dpsr = False
                    break          # visible illuminated terrain found

                if terrain_tan > highest_tan:
                    highest_tan = terrain_tan

            if not is_dpsr:
                break              # early exit: skip remaining directions

        if is_dpsr:
            result[i] = 1
    return result

# ── Warm-up: trigger JIT compilation (~20 s first time only) ─────────────────
print("Compiling Numba kernel …")
t_compile = time.perf_counter()
_e   = np.zeros((8,8), dtype=np.float32)
_ill = np.zeros((8,8), dtype=np.uint8)
_r   = np.array([3], dtype=np.int32)
_c   = np.array([3], dtype=np.int32)
classify_psr_pixels(_e, _ill, _r, _c,
                    ray_dr[:, :2], ray_dc[:, :2],
                    ray_dist[:, :2], np.minimum(ray_len, 2))
print(f"Kernel compiled in {time.perf_counter()-t_compile:.1f} s")


In [ ]:
print(f"Processing {len(psr_rows_all):,} PSR pixels …")
t0 = time.perf_counter()

dpsr_flags = classify_psr_pixels(
    elevation, illumination,
    psr_rows_all, psr_cols_all,
    ray_dr, ray_dc, ray_dist, ray_len,
)

elapsed = time.perf_counter() - t0
rate    = len(psr_rows_all) / max(elapsed, 1e-6)

print(f"  Done in     : {elapsed:.1f} s  ({elapsed/60:.1f} min)")
print(f"  Throughput  : {rate:,.0f} pixels/s")
print(f"  DPSR pixels : {dpsr_flags.sum():,}")
print(f"  Non-DPSR    : {(dpsr_flags==0).sum():,}")

# Reconstruct raster
dpsr_raster = np.zeros(elevation.shape, dtype=np.uint8)
dpsr_raster[psr_rows_all[dpsr_flags==1], psr_cols_all[dpsr_flags==1]] = 1

# Save
out_meta = DEM_META.copy(); out_meta.update(dtype="uint8", count=1, compress="lzw")
with rasterio.open(DPSR_PATH, "w", **out_meta) as dst:
    dst.write(dpsr_raster, 1)
print(f"  Saved → {DPSR_PATH}")


---

## Step 11 — DPSR Extraction (GPU — Numba CUDA) _(optional)_

Requires an NVIDIA GPU. Expected runtime: **30–120 seconds**.
Skip this cell if no GPU is available — Step 10 already produced `DPSR.tif`.


In [ ]:
try:
    from numba import cuda
    if not cuda.is_available():
        print("No CUDA GPU found — skipping. DPSR.tif from Step 10 is valid.")
    else:
        import math as _math

        THREADS = 256

        @cuda.jit(cache=True)
        def _gpu_kernel(elev, illum, rows, cols, dr, dc, dist, rlen, out):
            i = cuda.grid(1)
            if i >= rows.shape[0]: return
            nr = elev.shape[0]; nc = elev.shape[1]; na = dr.shape[0]
            row = rows[i]; col = cols[i]; cur_h = elev[row, col]
            is_dpsr = True
            for a in range(na):
                ht = -1.0e18; ns = rlen[a]
                for d in range(ns):
                    r = row + dr[a,d]; c = col + dc[a,d]
                    if r<0 or r>=nr or c<0 or c>=nc: break
                    tt = (elev[r,c] - cur_h) / dist[a,d]
                    if illum[r,c]==1 and tt>=ht: is_dpsr=False; break
                    if tt > ht: ht = tt
                if not is_dpsr: break
            out[i] = 1 if is_dpsr else 0

        print(f"GPU : {cuda.get_current_device().name.decode()}")
        print("Transferring arrays to GPU …")
        d_elev = cuda.to_device(elevation);      d_ill  = cuda.to_device(illumination)
        d_rows = cuda.to_device(psr_rows_all);   d_cols = cuda.to_device(psr_cols_all)
        d_dr   = cuda.to_device(ray_dr);         d_dc   = cuda.to_device(ray_dc)
        d_dist = cuda.to_device(ray_dist);       d_rlen = cuda.to_device(ray_len)
        d_out  = cuda.device_array(len(psr_rows_all), dtype=np.uint8)

        n_blocks = _math.ceil(len(psr_rows_all) / THREADS)
        print(f"Launching  blocks={n_blocks}  threads/block={THREADS} …")
        t0 = time.perf_counter()
        _gpu_kernel[n_blocks, THREADS](d_elev, d_ill, d_rows, d_cols,
                                        d_dr, d_dc, d_dist, d_rlen, d_out)
        cuda.synchronize()
        elapsed = time.perf_counter() - t0

        dpsr_flags_gpu = d_out.copy_to_host()
        dpsr_raster_gpu = np.zeros(elevation.shape, dtype=np.uint8)
        dpsr_raster_gpu[psr_rows_all[dpsr_flags_gpu==1], psr_cols_all[dpsr_flags_gpu==1]] = 1

        out_meta = DEM_META.copy(); out_meta.update(dtype="uint8",count=1,compress="lzw")
        with rasterio.open(DPSR_PATH, "w", **out_meta) as dst:
            dst.write(dpsr_raster_gpu, 1)

        print(f"  GPU done  : {elapsed:.1f} s  ({len(psr_rows_all)/elapsed:,.0f} px/s)")
        print(f"  DPSR px   : {dpsr_raster_gpu.sum():,}")
        print(f"  Saved → {DPSR_PATH}")
        dpsr_raster = dpsr_raster_gpu   # update for Step 12

except Exception as e:
    print(f"GPU skipped: {e}")


---

## Step 12 — Final Visualisation


In [ ]:
# Load final DPSR in case it was saved in a previous run
if "dpsr_raster" not in dir() or dpsr_raster is None:
    with rasterio.open(DPSR_PATH) as ds:
        dpsr_raster = ds.read(1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(psr_mask,     cmap="gray", interpolation="none")
axes[0].set_title(f"PSR Mask\n({psr_mask.sum():,} pixels)")
axes[0].axis("off")

axes[1].imshow(illumination, cmap="gray", interpolation="none")
axes[1].set_title(f"Illumination Map\n(az={SUN_AZIMUTH}°  el={SUN_ELEVATION}°)")
axes[1].axis("off")

axes[2].imshow(dpsr_raster,  cmap="hot",  interpolation="none")
axes[2].set_title(f"DPSR Output\n({dpsr_raster.sum():,} pixels)")
axes[2].axis("off")

plt.suptitle("DPSR Pipeline — Final Results", fontsize=14, fontweight="bold")
plt.tight_layout()
fig_path = IMG_DIR / "DPSR_final_summary.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Summary plot → {fig_path}")
print(f"DPSR raster  → {DPSR_PATH}")
print()
print(f"{'='*50}")
print(f"  PSR pixels   : {psr_mask.sum():,}")
print(f"  DPSR pixels  : {dpsr_raster.sum():,}")
print(f"  DPSR / PSR   : {dpsr_raster.sum()/max(psr_mask.sum(),1)*100:.2f} %")
print(f"{'='*50}")


## Step 13 — Spot-check Validation

A basic sanity check: a crater rim pixel should be able to see illuminated terrain (NOT DPSR); a deep interior pixel should not (IS DPSR).

Update the row/col values after inspecting the DEM in QGIS or via rasterio.


In [ ]:
# ── Spot-check validation ─────────────────────────────────────────────────────
# Update row/col after inspecting the DEM in QGIS.
# A rim pixel should see illuminated terrain (NOT DPSR).
# A deep interior pixel should not (IS DPSR).
print("Spot-check validation:")
checks = [
    ("Shackleton rim (expect NOT DPSR)",  7580, 7580, False),
    ("Shackleton interior (expect DPSR)", 7584, 7584, True),
]
for label, row, col, expect in checks:
    d_val = bool(dpsr_raster[row, col])
    ok = (d_val == expect)
    print(f"  [{'OK' if ok else 'CHECK'}] {label}")
    print(f"         elev={elevation[row,col]:.0f}m  "
          f"illum={illumination[row,col]}  "
          f"psr={psr_mask[row,col]}  dpsr={int(d_val)}  "
          f"(expected dpsr={int(expect)})")
